# Train `dataset2` in Google Colab

This notebook prepares and trains the dataset at `datasets/dataset2` using YOLOv8.

Google Drive folder: https://drive.google.com/drive/folders/1YHFQj6lBBv4aFS9i4XEQmv_gR5nvEjLZ

It also repairs the class IDs in the label files using `metadata.csv`, because the raw YOLO labels in this dataset do not currently line up with the 9 weapon categories listed in the metadata.

## 1. Install dependencies

In [1]:
import sys

# Use the notebook kernel's Python explicitly so installs land in the active environment
!{sys.executable} -m pip install -q -U ultralytics pandas pyyaml

import ultralytics
print('ultralytics version:', ultralytics.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 287.9 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 321.2 kB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=v

## 2. Mount Google Drive and locate the dataset

Before running this cell, upload your project folder to Google Drive or upload just `dataset2` to Colab.

Provided Drive folder:
- https://drive.google.com/drive/folders/1YHFQj6lBBv4aFS9i4XEQmv_gR5nvEjLZ

Expected dataset structure:
- `dataset2/metadata.csv`
- `dataset2/weapon_detection/train/images`
- `dataset2/weapon_detection/train/labels`
- `dataset2/weapon_detection/val/images`
- `dataset2/weapon_detection/val/labels`

In [2]:
from pathlib import Path

DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1YHFQj6lBBv4aFS9i4XEQmv_gR5nvEjLZ'
print('Drive folder URL:', DRIVE_FOLDER_URL)

IN_COLAB = False
FORCE_REMOUNT = False  # Set True if Drive appears mounted but points to the wrong account.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=FORCE_REMOUNT)
    IN_COLAB = True
except Exception as exc:
    print(f'Colab Drive mount skipped: {exc}')

# Optional manual override if auto-discovery cannot find your dataset folder.
# Example: MANUAL_DATASET_ROOT = '/content/drive/MyDrive/Night Surveillance System - Final1/datasets/dataset2'
MANUAL_DATASET_ROOT = None

def is_valid_dataset_root(path: Path) -> bool:
    return (
        path.exists()
        and (path / 'metadata.csv').exists()
        and (path / 'weapon_detection' / 'train' / 'images').exists()
        and (path / 'weapon_detection' / 'train' / 'labels').exists()
        and (path / 'weapon_detection' / 'val' / 'images').exists()
        and (path / 'weapon_detection' / 'val' / 'labels').exists()
    )

candidate_paths = []
if MANUAL_DATASET_ROOT:
    candidate_paths.append(Path(MANUAL_DATASET_ROOT))

candidate_paths.extend([
    Path('/content/drive/MyDrive/Night Surveillance System - Final1/datasets/dataset2'),
    Path('/content/drive/MyDrive/Night Surveillance System - Final1/Night Surveillance System - Final1/datasets/dataset2'),
    Path('/content/drive/MyDrive/dataset2'),
    Path('/content/dataset2'),
    Path('/content/Night Surveillance System - Final1/datasets/dataset2'),
])

for drive_root in [
    Path('/content/drive/MyDrive'),
    Path('/content/drive/Shareddrives'),
    Path('/content/drive/.shortcut-targets-by-id'),
]:
    if not drive_root.exists():
        continue
    candidate_paths.extend([
        drive_root / 'dataset2',
        drive_root / 'Night Surveillance System - Final1' / 'datasets' / 'dataset2',
        drive_root / 'Night Surveillance System - Final1' / 'Night Surveillance System - Final1' / 'datasets' / 'dataset2',
    ])

DATASET_ROOT = next((path for path in candidate_paths if is_valid_dataset_root(path)), None)

if DATASET_ROOT is None and IN_COLAB:
    print('Auto path check failed. Searching Google Drive for dataset2...')
    search_roots = [
        Path('/content/drive/MyDrive'),
        Path('/content/drive/Shareddrives'),
        Path('/content/drive/.shortcut-targets-by-id'),
    ]
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for metadata_file in search_root.rglob('metadata.csv'):
            possible_root = metadata_file.parent
            if possible_root.name != 'dataset2':
                continue
            if is_valid_dataset_root(possible_root):
                DATASET_ROOT = possible_root
                break
        if DATASET_ROOT is not None:
            break

if DATASET_ROOT is None:
    mydrive = Path('/content/drive/MyDrive')
    mydrive_dirs = []
    if mydrive.exists():
        mydrive_dirs = sorted([p.name for p in mydrive.iterdir() if p.is_dir()])[:25]
    raise FileNotFoundError(
        'Could not find dataset2 automatically.\n'
        'Fix options:\n'
        '1) Open the Drive link and click "Add shortcut to Drive" (MyDrive), OR\n'
        '2) Set MANUAL_DATASET_ROOT to your exact dataset2 path.\n'
        f'Visible MyDrive folders (first 25): {mydrive_dirs}'
    )

print('Using dataset root:', DATASET_ROOT)

Mounted at /content/drive
Using dataset root: /content/drive/MyDrive/dataset2


## 3. Repair labels and build a clean YOLO dataset

In [3]:
import shutil
import pandas as pd
import yaml
from collections import Counter

metadata_path = DATASET_ROOT / 'metadata.csv'
raw_root = DATASET_ROOT / 'weapon_detection'

if not metadata_path.exists():
    raise FileNotFoundError(f'Missing metadata file: {metadata_path}')
if not raw_root.exists():
    raise FileNotFoundError(f'Missing weapon_detection folder: {raw_root}')

df = pd.read_csv(metadata_path)
required_cols = {'imagefile', 'labelfile', 'target'}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f'metadata.csv is missing columns: {sorted(missing_cols)}')

df['target'] = df['target'].astype(int)
df['class_name'] = df['imagefile'].str.rsplit('_', n=1).str[0]

target_to_name = (
    df[['target', 'class_name']]
    .drop_duplicates()
    .sort_values('target')
)
class_map = dict(zip(target_to_name['target'], target_to_name['class_name']))
class_names = [class_map[idx] for idx in sorted(class_map)]

print('Class mapping:')
for idx, name in enumerate(class_names):
    print(f'  {idx}: {name}')

label_to_target = dict(zip(df['labelfile'], df['target']))
image_to_target = dict(zip(df['imagefile'], df['target']))

prepared_root = Path('/content/prepared_dataset2_yolo')
if prepared_root.exists():
    shutil.rmtree(prepared_root)

for split in ['train', 'val']:
    (prepared_root / 'images' / split).mkdir(parents=True, exist_ok=True)
    (prepared_root / 'labels' / split).mkdir(parents=True, exist_ok=True)

split_counts = {}
box_counts = Counter()

for split in ['train', 'val']:
    image_dir = raw_root / split / 'images'
    label_dir = raw_root / split / 'labels'
    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(f'Missing split folders for {split}: {image_dir} or {label_dir}')

    image_count = 0
    for image_path in sorted(image_dir.iterdir()):
        if not image_path.is_file():
            continue

        label_path = label_dir / f'{image_path.stem}.txt'
        if not label_path.exists():
            raise FileNotFoundError(f'Missing label for image: {image_path.name}')

        target = label_to_target.get(label_path.name)
        if target is None:
            target = image_to_target.get(image_path.name)
        if target is None:
            raise KeyError(f'No target mapping found in metadata for {image_path.name}')

        repaired_lines = []
        for raw_line in label_path.read_text(encoding='utf-8', errors='ignore').splitlines():
            parts = raw_line.strip().split()
            if len(parts) < 5:
                continue
            repaired_lines.append(f"{target} {' '.join(parts[1:5])}")
            box_counts[int(target)] += 1

        if not repaired_lines:
            continue

        shutil.copy2(image_path, prepared_root / 'images' / split / image_path.name)
        (prepared_root / 'labels' / split / label_path.name).write_text(
            '\n'.join(repaired_lines) + '\n',
            encoding='utf-8'
        )
        image_count += 1

    split_counts[split] = image_count

dataset_yaml = {
    'path': str(prepared_root),
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(class_names),
    'names': class_names,
}

yaml_path = prepared_root / 'dataset.yaml'
yaml_path.write_text(yaml.safe_dump(dataset_yaml, sort_keys=False), encoding='utf-8')

print('\nPrepared dataset written to:', prepared_root)
print('dataset.yaml:', yaml_path)
print('images per split:', split_counts)
print('boxes per class:')
for idx, name in enumerate(class_names):
    print(f'  {idx}: {name} -> {box_counts[idx]}')

Class mapping:
  0: Automatic Rifle
  1: Bazooka
  2: Grenade Launcher
  3: Handgun
  4: Knife
  5: Shotgun
  6: SMG
  7: Sniper
  8: Sword

Prepared dataset written to: /content/prepared_dataset2_yolo
dataset.yaml: /content/prepared_dataset2_yolo/dataset.yaml
images per split: {'train': 571, 'val': 143}
boxes per class:
  0: Automatic Rifle -> 145
  1: Bazooka -> 81
  2: Grenade Launcher -> 104
  3: Handgun -> 159
  4: Knife -> 158
  5: Shotgun -> 118
  6: SMG -> 145
  7: Sniper -> 115
  8: Sword -> 130


## 4. Train YOLOv8

In [4]:
from pathlib import Path
import sys
import subprocess

try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    print('ultralytics not found in current kernel. Installing now...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics'])
    from ultralytics import YOLO

WEIGHTS = 'yolov8n.pt'
EPOCHS = 50
IMGSZ = 640
BATCH = 16
RUN_NAME = 'dataset2_weapon_detection'

model = YOLO(WEIGHTS)
train_results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project='/content/runs',
    name=RUN_NAME,
    patience=10,
    workers=2,
    verbose=True,
)

print('Training finished.')
print('Best weights should be at:', Path('/content/runs') / RUN_NAME / 'weights' / 'best.pt')

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/prepared_dataset2_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dataset2_weapon_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_

## 5. Validate the trained model

In [5]:
best_weights = Path('/content/runs') / RUN_NAME / 'weights' / 'best.pt'
best_model = YOLO(best_weights)
metrics = best_model.val(data=str(yaml_path))
metrics

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2155.1±1643.3 MB/s, size: 302.9 KB)
val: Scanning /content/prepared_dataset2_yolo/labels/val.cache... 143 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 143/143 54.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.3s/it 20.8s2.6s
                   all        143        219      0.866      0.825      0.864       0.69
       Automatic Rifle         15         20      0.677       0.75       0.73      0.572
               Bazooka         13         16      0.646      0.562      0.584      0.438
      Grenade Launcher         18         24      0.825      0.985       0.97      0.738
               Handgun         14         38      0.891      0.862      0.934       0.75
                 Knife         14        

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fbedd55cb30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0470

## 6. Download the trained weights

In [7]:
from pathlib import Path
import shutil

best_weights = Path('/content/runs') / RUN_NAME / 'weights' / 'best.pt'
print('best.pt path:', best_weights)

if not best_weights.exists():
    raise FileNotFoundError(f'Weights file not found: {best_weights}')

if IN_COLAB:
    from google.colab import files

    # Save a copy inside your Drive project folder so it can be accessed from a link later.
    if '/content/drive' in str(DATASET_ROOT):
        project_root = DATASET_ROOT.parent.parent if DATASET_ROOT.parent.name == 'datasets' else DATASET_ROOT.parent
    else:
        project_root = Path('/content/drive/MyDrive/Night Surveillance System - Final1')

    export_dir = project_root / 'trained_weights'
    export_dir.mkdir(parents=True, exist_ok=True)
    drive_best = export_dir / f'{RUN_NAME}_best.pt'
    shutil.copy2(best_weights, drive_best)

    print('Copied weights to Drive:', drive_best)
    print('Drive folder URL:', DRIVE_FOLDER_URL)
    print('Open the link above, then open trained_weights to download the .pt file.')

    # Also trigger direct browser download from Colab runtime.
    files.download(str(best_weights))
else:
    print('Not running in Colab, so automatic download was skipped.')

best.pt path: /content/runs/dataset2_weapon_detection/weights/best.pt
Copied weights to Drive: /content/drive/MyDrive/trained_weights/dataset2_weapon_detection_best.pt
Drive folder URL: https://drive.google.com/drive/folders/1YHFQj6lBBv4aFS9i4XEQmv_gR5nvEjLZ
Open the link above, then open trained_weights to download the .pt file.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>